In [2]:
import sys
import os
import polars as pl
import numpy as np
# Add the project root directory to PYTHONPATH
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
sys.path.insert(0, project_root)

# Import necessary modules
import groundinsight as gi
from groundinsight.models.core_models import BusType, BranchType

In [20]:
#Test the Setting of mesh strucutre with multiple cables 

rho = 100

# Create a network
net = gi.create_network(name="MyTestNetwork", frequencies=[50, 250, 350])
net.description = "CIRED example Network"

bus_type = BusType(
    name="BusTypeFormulaTest",
    description="Example bus type with parameters",
    system_type="Grounded",
    voltage_level=20,
    impedance_formula="0.01201149 * rho + 0.00024195 * f - 0.00000034 * rho*f + 0.01444725 + 1j * (-0.00003812 * rho + 0.00030091 * f + 0.00000089*rho*f + 0.01997045)",
)

bus_type_uw = BusType(
    name="BusTypeFormulaTestUW",
    description="Example bus type with parameters",
    system_type="Grounded",
    voltage_level=20,
    impedance_formula="0.2"
    )

self_impedance_formula = "(0.24/1000 + (2*pi*f * 4*pi*10**(-7) / 8) + 1j * (2*pi*f * 4*pi*10**(-7) / (2 * pi)) * log((1.8514 / sqrt(2 * pi * f * 4*pi*10**(-7) / rho)) / (0.040**2 * 0.035)**(1/3)))*l"
mutual_impedance_formula = "((2*pi*f * 4*pi*10**(-7) / 8) + 1j * (2*pi*f * 4*pi*10**(-7) / (2 * pi)) * log((1.8514 / sqrt(2 * pi * f * 4*pi*10**(-7) / rho)) / (0.040**2 * 0.035)**(1/3)))*l"

branch_type = BranchType(
    name="TestBranchType",
    description="A test branch type",
    grounding_conductor=True,
    self_impedance_formula=self_impedance_formula,
    mutual_impedance_formula=mutual_impedance_formula
)

branch_ohl = BranchType(
    name="OHLine",
    description="An overhead line",
    grounding_conductor=False,
    self_impedance_formula="NaN",
    mutual_impedance_formula="NaN"
)

number_buses = 11
#create buses with a for loop
for i in range(1, number_buses):
    gi.create_bus(name=f"bus{i}", type=bus_type, network=net, specific_earth_resistance=rho)


#create branch 
#defining a line length of each branch
line_length = 1000

#create bracnches with a for loop
for i in range(1, number_buses-1):
    gi.create_branch(name=f"branch{i}", type=branch_type, from_bus=f"bus{i}", to_bus=f"bus{i+1}", length=line_length, specific_earth_resistance=rho, network=net)

#create addition branch
gi.create_branch(name=f"additional_branch", type=branch_type, from_bus=f"bus1", to_bus=f"bus3", length=line_length, specific_earth_resistance=rho, network=net)

#create a faults
fault_scaling = {50: 1.0, 250: 1, 350:1}
faultnames = []
for i in range(1, number_buses):
    gi.create_fault(name=f"fault{i}", bus=f"bus{i}", description="A fault at bus {i}", scalings=fault_scaling, network=net)
    faultnames.append(f"fault{i}")


#soruce currents
source_values = {50:60, 250:60, 350:60}

#add a source at bus1
source = gi.create_source(name="source1", bus="bus1", values=source_values, network=net)

#define the paths of the network
gi.create_paths(network=net)

#for loop over each fault
for i in range(number_buses-1):
    # Run fault calculations
    gi.run_fault(net, fault_name=f"fault{i+1}")


all_imp = net.res_all_impedances()
all_imp = all_imp.filter(pl.col("frequency_Hz") == 50)

all_imp




fault_name,fault_bus,frequency_Hz,grounding_impedance_Ohm,grounding_impedance_deg,reduction_factor
str,str,f64,f64,f64,f64
"""fault1""","""bus1""",50.0,null,null,null
"""fault2""","""bus2""",50.0,0.211958,50.00995,1.36967
"""fault3""","""bus3""",50.0,0.156115,-73.944523,0.950063
"""fault4""","""bus4""",50.0,0.352114,-50.314365,0.367579
"""fault5""","""bus5""",50.0,0.427917,-40.085365,0.317822
"""fault6""","""bus6""",50.0,0.451261,-36.316888,0.316062
"""fault7""","""bus7""",50.0,0.452716,-35.375166,0.323631
"""fault8""","""bus8""",50.0,0.449785,-37.015154,0.331607
"""fault9""","""bus9""",50.0,0.48371,-42.021582,0.337494


In [ ]:
#Testcase 2: two sources at the beginnging of the network and at the end of the network